In [1]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [ ]:
%LoadCheckpoint /home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/small_bench/checkpoints/pre_cell_1.pickle

In [3]:
%%RecordEvent
%%time
### cell 1 ###

# Pre-bind heavy-lookups
_read_csv, _to_csv, _exists = pd.read_csv, pd.DataFrame.to_csv, os.path.exists

def load_survey_data(file_path, year, skip):
    df = _read_csv(
        file_path,
        low_memory=False,
        encoding='ISO-8859-1',
        skiprows=skip
    )
    out_file = f'kaggle_survey_{year}.csv'
    if not _exists(out_file):
        _to_csv(df, out_file, index=False)
    return df

# Sampling factor
factor = 1

# Create chart subdirectories in one go (root will be created implicitly)
root = (
    '/home/dias-benchmarks/notebooks/paultimothymooney/'
    'kaggle-survey-2022-all-results/kaggle/working/individual_charts'
)
for sub in ('data', 'charts'):
    os.makedirs(f'{root}/{sub}', exist_ok=True)

# Base path for all survey inputs
base_prefix = (
    '/home/dias-benchmarks/notebooks/paultimothymooney/'
    'kaggle-survey-2022-all-results/input/kaggle-survey-'
)

# (year, filename, skiprows)
files = [
    (2017, 'multipleChoiceResponses.csv', 0),
    (2018, 'multipleChoiceResponses.csv', 1),
    (2019, 'multiple_choice_responses.csv', 1),
    (2020, 'kaggle_survey_2020_responses.csv', 1),
    (2021, 'kaggle_survey_2021_responses.csv', 1),
    (2022, 'kaggle_survey_2022_responses.csv', 1),
]

# Load & sample each year in a single comprehension
(
    responses_df_2017,
    responses_df_2018,
    responses_df_2019,
    responses_df_2020,
    responses_df_2021,
    responses_df_2022,
) = [
    load_survey_data(
        f'{base_prefix}{year}/{fname}',
        year,
        skip
    )
    .sample(frac=factor, random_state=0)
    for year, fname, skip in files
]

CPU times: user 4.51 s, sys: 956 ms, total: 5.47 s
Wall time: 5.46 s


In [ ]:
%Checkpoint /home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten_cpu/o4_mini_high/checkpoints/post_cell_1_try_4.pickle